# 🍪 Cookie Experiment — Pilot-Based Power Analysis

### Goal
Estimate the observed effect size (Cohen’s d_z) for our pilot dataset and compute the required number of cookie pairs 
for a paired t-test at α = 0.05 and 80% power.

### Dataset description
Each `pair_id` represents a matched pair of cookies baked under identical conditions, 
with one in the **toaster oven (`t`)** and one in the **regular oven (`r`)**.  
The main dependent variable (DV) is `weight_loss_pct`, representing water loss as a percentage of initial weight.

In [1]:
# 1) Imports & setup

import pandas as pd
import numpy as np
from statsmodels.stats.power import TTestPower
import math

# parameters
PATH = "../data/raw/cookie_pairs_pilot.csv"
ALPHA = 0.05
TARGET_POWER = 0.80
BUFFER = 0.10  # 10% extra safety margin

In [2]:
# 2) Read the pilot data
df = pd.read_csv(PATH, encoding="utf-8-sig")

# preview data
display(df.head())

# standardize column names
df.columns = [c.strip().lower() for c in df.columns]

,pair_id,batch_id,randomization_order,preheat_f,bake_time_min,cool_down_min,init_weight_g,d1_cm,d2_cm,d_avg,thickness_cm,spread_ratio,post_weight_g,weight_loss_pct
0,1,1,t,375,8,3,30,8.2,8.4,8.30,0.6,13.833333,27.9,0.070000
1,1,1,r,375,8,3,30,7.9,8.4,8.15,0.5,16.300000,27.1,0.096667
2,2,1,t,375,8,3,30,7.9,8.1,8.00,0.4,20.000000,28.5,0.050000
3,2,1,r,375,8,3,30,8.5,7.9,8.20,0.6,13.666667,26.7,0.110000
4,3,1,t,375,8,3,30,8.0,7.4,7.70,0.3,25.666667,27.5,0.083333


## 1. Power Analysis Based on Weight Loss Ratio

In [3]:
# 3) Pivot into pairs (toaster vs regular)

# pivot so that each row = one pair
pairs_1 = df.pivot(index="pair_id", columns="randomization_order", values="weight_loss_pct")

pairs_1.head()

randomization_order,r,t
pair_id,,
1,0.096667,0.070000
2,0.110000,0.050000
3,0.060000,0.083333
4,0.096667,0.010000
5,0.083333,0.053333


In [4]:
# 4) Compute paired differences (toaster − regular)

diffs = (pairs_1["t"].astype(float) - pairs_1["r"].astype(float)).values
n_pilot = len(diffs)
mean_diff = np.mean(diffs)
sd_diff = np.std(diffs, ddof=1)

print(f"[pilot] n_pairs = {n_pilot}")
print(f"[pilot] mean_diff = {mean_diff:.4f}  (toaster - regular)")
print(f"[pilot] sd_diff   = {sd_diff:.4f}")

[pilot] n_pairs = 7
[pilot] mean_diff = -0.0352  (toaster - regular)
[pilot] sd_diff   = 0.0363


In [5]:
# 5) Compute effect size (Cohen’s d_z) and required sample size

dz = abs(mean_diff) / sd_diff
tt = TTestPower()
n_exact = tt.solve_power(effect_size=dz, alpha=ALPHA, power=TARGET_POWER, alternative="two-sided")
required_pairs = math.ceil(n_exact)
planning_pairs = math.ceil(required_pairs / (1 - BUFFER))

print(f"[effect] Cohen's d_z = {dz:.3f}")
print(f"[power ] Required pairs @80% power = {required_pairs} (~{required_pairs*2} cookies)")
print(f"[plan  ] With 10% buffer → {planning_pairs} pairs (~{planning_pairs*2} cookies)")

[effect] Cohen's d_z = 0.972
[power ] Required pairs @80% power = 11 (~22 cookies)
[plan  ] With 10% buffer → 13 pairs (~26 cookies)


## 2. Power Analysis Based on Spread Ratio

Spread Ratio = Average Diameter / Thickness

In [6]:
pairs_2 = df.pivot(index="pair_id", columns="randomization_order", values="spread_ratio")

pairs_2.head()

randomization_order,r,t
pair_id,,
1,16.300000,13.833333
2,13.666667,20.000000
3,13.416667,25.666667
4,15.600000,15.800000
5,10.187500,16.200000


In [7]:
diffs = (pairs_2["t"].astype(float) - pairs_2["r"].astype(float)).values
n_pilot = len(diffs)
mean_diff = np.mean(diffs)
sd_diff = np.std(diffs, ddof=1)

print(f"[pilot] n_pairs = {n_pilot}")
print(f"[pilot] mean_diff = {mean_diff:.4f}  (toaster - regular)")
print(f"[pilot] sd_diff   = {sd_diff:.4f}")

[pilot] n_pairs = 7
[pilot] mean_diff = 4.6113  (toaster - regular)
[pilot] sd_diff   = 5.3041


In [8]:
dz = abs(mean_diff) / sd_diff
tt = TTestPower()
n_exact = tt.solve_power(effect_size=dz, alpha=ALPHA, power=TARGET_POWER, alternative="two-sided")
required_pairs = math.ceil(n_exact)
planning_pairs = math.ceil(required_pairs / (1 - BUFFER))

print(f"[effect] Cohen's d_z = {dz:.3f}")
print(f"[power ] Required pairs @80% power = {required_pairs} (~{required_pairs*2} cookies)")
print(f"[plan  ] With 10% buffer → {planning_pairs} pairs (~{planning_pairs*2} cookies)")

[effect] Cohen's d_z = 0.869
[power ] Required pairs @80% power = 13 (~26 cookies)
[plan  ] With 10% buffer → 15 pairs (~30 cookies)


# ✅ Summary — Pilot-Based Power Analysis Results

### Overview
This pilot-based power analysis estimated the effect sizes for our two main dependent variables — **Weight Loss %** (primary) and **Spread Ratio** (secondary) — to determine the sample size required for 80% statistical power under α = 0.05.  
Since no comparable literature was available, our pilot dataset served as an empirical basis for effect size estimation.

---

### Results Summary

| Metric | Cohen’s dₙ | Required Pairs (80% Power) | Total Cookies | +10% Buffer (Planned Pairs) | Planned Cookies |
|:-------|:------------:|:---------------------------:|:--------------:|:-----------------------------:|:----------------:|
| **Weight Loss %** | 0.972 | 11 | 22 | 13 | 26 |
| **Spread Ratio**  | 0.869 | 13 | 26 | 15 | 30 |

---

### Interpretation
- Both dependent variables show **large effect sizes** (dₙ ≈ 0.9), indicating a clear and consistent difference between toaster and regular oven cookies.  
- For the **primary variable (weight loss %)**, only about **11 pairs (~22 cookies)** are needed to achieve 80% power; with a 10% buffer, the planned number was **13 pairs (~26 cookies)**.  
- For **spread ratio**, a slightly larger but still manageable sample of **15 pairs (~30 cookies)** was sufficient.  
- In our confirmatory experiment, we ultimately baked **28 pairs of cookies (54 total)**, exceeding the required sample size and ensuring strong statistical reliability.

---

### Conclusion
The pilot-based power analysis provided a realistic, data-driven foundation for sample size planning.  
With observed effect sizes around **dₙ ≈ 0.9**, the final dataset of **28 pairs** far exceeds the minimum power requirement, ensuring that our analysis can confidently detect meaningful differences in cookie properties between toaster and regular ovens.